<a href="https://colab.research.google.com/github/ADRIANVM117/data-science-portfolio/blob/main/BROWNIAN_GEOME_MOV.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Si estás en un entorno nuevo (Colab, por ejemplo), primero instala yfinance:
# !pip install yfinance --upgrade --quiet

import datetime as dt
import yfinance as yf
import pandas as pd

# 1. Definimos fechas: desde el inicio del año hasta hoy
start_date = "2025-01-01"
end_date = dt.date.today().strftime("%Y-%m-%d")

# 2. Definimos el ticker de Tesla
ticker = "TSLA"

# 3. Descargamos los datos desde Yahoo Finance
tesla_df = yf.download(ticker, start=start_date, end=end_date)

# 4. Vemos las primeras filas para confirmar que todo está bien
print(tesla_df.head())

# (Opcional) nos quedamos sólo con el precio de cierre ajustado para el GBM
tesla_close = tesla_df["Close"].copy()
print("\nSerie de precios (Adj Close):")
print(tesla_close.head())


/tmp/ipython-input-1888415824.py:16: FutureWarning: YF.download() has changed argument auto_adjust default to True
  tesla_df = yf.download(ticker, start=start_date, end=end_date)
[*********************100%***********************]  1 of 1 completed

Price            Close        High         Low        Open     Volume
Ticker            TSLA        TSLA        TSLA        TSLA       TSLA
Date                                                                 
2025-01-02  379.279999  392.730011  373.040009  390.100006  109710700
2025-01-03  410.440002  411.880005  379.450012  381.480011   95423300
2025-01-06  411.049988  426.429993  401.700012  423.200012   85516500
2025-01-07  394.359985  414.329987  390.000000  405.829987   75699500
2025-01-08  394.940002  402.500000  387.399994  392.950012   73038800

Serie de precios (Adj Close):
Ticker            TSLA
Date                  
2025-01-02  379.279999
2025-01-03  410.440002
2025-01-06  411.049988
2025-01-07  394.359985
2025-01-08  394.940002


# <b> Modelando precios mediante un movimiento browniano geométrico  </b>




## <b> CASO I:  </b> Volatilidad constante

In [ ]:
# calcular los retornos logaritmicos | mu y std constante
log_retornos = tesla_df['Close'].pct_change().dropna()
mu = log_retornos.mean()
std = log_retornos.std()

##### SIMULACION ############
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import scipy.stats as stats

# Simulación
s_0 = tesla_df['Close'].iloc[-1] # ultimo precio observado de tesla

n_steps = 10 # numero de dias que busco simular
n_sims = 100000 # numero de trayectorias (escenarios)
dt = 1   # son pasos diarios, 1 dia

# matriz para guardar trayectorias
trayectorias = np.zeros((n_steps + 1, n_sims))
trayectorias[0,:] = s_0

# generamos shocks noramales. ¿Para que? : dentro de una GBM son numeros aleatorios que siguen una normal(0,1) : representado ruido del mercado o comportamineto estocastico del movimiento browniano
z = np.random.normal(size = (n_steps+1, n_sims)) # cada shock para n dia y n simulacion

# simular trayectorias usando la solución de una GBM
for t in range(1, n_steps+1):
  trayectorias[t,:] = trayectorias[t-1, :] * np.exp((mu.item() - 0.5 * std.item() **2) * dt + std.item() * np.sqrt(dt) * z[t-1,:])

# Pasar a un df con indice de fechas futuro
ultimo_dia = tesla_df.index[-1]
fecha_futuro = pd.bdate_range(start = ultimo_dia, periods = n_steps + 1)
trayectorias_df = pd.DataFrame(trayectorias, index = fecha_futuro)
trayectorias_df.head()

,0,1,2,3,4,5,6,7,8,9,...,99990,99991,99992,99993,99994,99995,99996,99997,99998,99999
2025-11-19,403.989990,403.989990,403.989990,403.989990,403.989990,403.989990,403.989990,403.989990,403.989990,403.989990,...,403.989990,403.989990,403.989990,403.989990,403.989990,403.989990,403.989990,403.989990,403.989990,403.989990
2025-11-20,423.994109,415.895450,391.194260,419.347953,412.460263,403.433139,418.404509,400.708050,391.304897,390.548172,...,412.101732,406.332064,381.548574,390.772350,425.178368,386.720163,392.261988,418.936059,404.923601,396.725744
2025-11-21,420.512413,434.740281,396.439195,418.537242,398.481136,406.915407,411.368818,439.785986,402.684694,387.888981,...,407.228523,403.121913,382.388838,394.896457,453.956609,395.183687,373.210159,403.820160,425.503550,397.235594
2025-11-24,409.832068,449.208065,419.664822,398.574512,380.493579,390.071394,414.127374,427.656602,429.010125,385.515307,...,406.930885,412.760427,364.168375,416.712262,461.447501,412.520336,361.217227,400.186369,434.037019,425.818066
2025-11-25,404.708141,437.647138,398.603516,387.122167,369.320463,385.777722,406.723492,421.713227,424.552449,381.165016,...,438.163872,400.442926,357.457603,410.027078,469.038971,414.684256,369.610334,384.922922,459.374781,440.608428


## <b> Graficando </b>
- Graficando algunas trayectorias
- Viendo trayectoria media y percentiles: Ver <b>Escenario promedio y 'abanico' de riesgos  </b>

In [ ]:
import plotly.graph_objects as go

# Para no saturar la gráfica, tomamos por ejemplo 50 trayectorias
n_to_plot = 50
cols_to_plot = trayectorias_df.columns[:n_to_plot]

fig = go.Figure()

for col in cols_to_plot:
    fig.add_trace(
        go.Scatter(
            x=trayectorias_df.index,
            y=trayectorias_df[col],
            mode="lines",
            line=dict(width=1),
            showlegend=False,
            hoverinfo="skip"  # opcional: evita demasiados tooltips
        )
    )

fig.update_layout(
    title="Simulaciones GBM del precio de Tesla (trayectorias simuladas) : Volatilidad constante",
    xaxis_title="Fecha",
    yaxis_title="Precio simulado",
    template="plotly_white"
)

fig.show()


In [ ]:
# Estadísticos por fecha
median_path = trayectorias_df.median(axis=1)
p10 = trayectorias_df.quantile(0.10, axis=1)
p90 = trayectorias_df.quantile(0.90, axis=1)

fig = go.Figure()

# Banda P10–P90
fig.add_trace(
    go.Scatter(
        x=trayectorias_df.index,
        y=p90,
        mode="lines",
        line=dict(width=0),
        showlegend=False,
        hoverinfo="skip"
    )
)
fig.add_trace(
    go.Scatter(
        x=trayectorias_df.index,
        y=p10,
        mode="lines",
        fill="tonexty",  # rellena entre p10 y p90
        line=dict(width=0),
        name="P10–P90",
        hoverinfo="skip"
    )
)

# Trayectoria mediana
fig.add_trace(
    go.Scatter(
        x=trayectorias_df.index,
        y=median_path,
        mode="lines",
        name="Mediana",
        line=dict(width=2)
    )
)

fig.update_layout(
    title="Distribución de trayectorias GBM de Tesla (mediana y banda P10–P90)",
    xaxis_title="Fecha",
    yaxis_title="Precio simulado",
    template="plotly_white"
)

fig.show()


# <b> CASO II </b>: Volatilidad variable. Usando <B> GARCH(1,1) </B>  
- Con GARCH, Lo que hacemos es estimar un modelo que da una volatilidd distinta para cada dia futuro

In [ ]:
# Si no lo tienes instalado:
!pip install arch --quiet

from arch import arch_model

# 1. Serie de precios de cierre
precios = tesla_df['Close'].copy()

# 2. Log-retornos diarios
log_ret = np.log(precios / precios.shift(1)).dropna()

# 3. Drift: media de los log-retornos (diaria)
mu = log_ret.mean()

# 4. Ajustamos un GARCH(1,1) a los log-retornos
# multiplicamos por 100 para que la escala sea en porcentaje
am = arch_model(log_ret * 100, vol='Garch', p=1, q=1, dist='normal')
res = am.fit(disp='off')   # disp='off' para que no imprima el resumen

# 5. Pronóstico de volatilidad condicional para los próximos n_steps días
n_steps = 10
forecast = res.forecast(horizon=n_steps)

# 6. Extraemos la varianza pronosticada (última fila, horizonte futuro)
# forecast.variance.values tiene shape (n_obs+1, horizon)
var_forecast = forecast.variance.values[-1, :]  # array de largo n_steps
sigma_path = np.sqrt(var_forecast) / 100        # regresamos a "escala original"


# --- Parámetros de simulación ---
s_0 = precios.iloc[-1]   # último precio observado de Tesla
n_sims = 100000
dt = 1.0                 # pasos diarios

# --- Matriz de trayectorias ---
trayectorias = np.zeros((n_steps + 1, n_sims))
trayectorias[0, :] = s_0

# --- Shocks normales (ruido browniano) ---
z = np.random.normal(size=(n_steps, n_sims))  # n_steps x n_sims

# --- Simulación GBM con volatilidad variable σ_t (GARCH) ---
for t in range(1, n_steps + 1):
    sigma_t = sigma_path[t-1]           # volatilidad para el día t
    drift_t = mu.item() - 0.5 * sigma_t**2     # drift ajustado por σ_t
    trayectorias[t, :] = trayectorias[t-1, :] * np.exp(
        drift_t * dt + sigma_t * np.sqrt(dt) * z[t-1, :]
    )

# --- Pasar a DataFrame con índice de fechas ---
ultimo_dia = tesla_df.index[-1]
fechas_futuras = pd.bdate_range(start=ultimo_dia, periods=n_steps + 1)

trayectorias_df = pd.DataFrame(trayectorias, index=fechas_futuras)
trayectorias_df.head()



,0,1,2,3,4,5,6,7,8,9,...,99990,99991,99992,99993,99994,99995,99996,99997,99998,99999
2025-11-19,403.989990,403.989990,403.989990,403.989990,403.989990,403.989990,403.989990,403.989990,403.989990,403.989990,...,403.989990,403.989990,403.989990,403.989990,403.989990,403.989990,403.989990,403.989990,403.989990,403.989990
2025-11-20,413.575639,413.615218,415.874666,382.600995,380.585136,402.970520,414.079611,381.727400,397.393379,380.836655,...,384.386997,422.110249,399.878179,408.297158,390.577700,373.831587,428.627430,391.125653,412.951940,399.195198
2025-11-21,410.513164,433.037407,415.859337,379.257481,376.456872,418.880288,402.609880,369.911912,382.357053,400.825131,...,366.254007,432.610499,386.082316,416.305049,370.150010,373.068136,434.591723,364.658656,426.980584,406.343698
2025-11-24,403.679326,441.597715,437.028497,375.277312,380.725598,428.652572,398.688659,382.129154,397.252794,387.450106,...,371.976261,438.504487,372.892501,399.327057,381.546178,389.432399,436.371814,366.643039,425.687811,402.238056
2025-11-25,408.844099,417.153396,444.419937,394.030527,392.218574,435.859470,384.186268,359.769274,429.060899,374.127553,...,347.708588,431.988559,352.670805,392.066996,397.156897,387.943432,442.456906,373.904423,423.906722,391.839419


In [ ]:
n_to_plot = 50
cols_to_plot = trayectorias_df.columns[:n_to_plot]

fig = go.Figure()

for col in cols_to_plot:
    fig.add_trace(
        go.Scatter(
            x=trayectorias_df.index,
            y=trayectorias_df[col],
            mode="lines",
            line=dict(width=1),
            showlegend=False
        )
    )

fig.update_layout(
    title="Simulaciones GBM de Tesla con volatilidad GARCH(1,1)",
    xaxis_title="Fecha",
    yaxis_title="Precio simulado",
    template="plotly_white"
)

fig.show()


# <b> EJERCICIO II: </b> Por medio de la ecuación de Feynman Kac y el valor esperado, cal-cular el precio de una opción europea que tenga como subyacente las acciones de Tesla.

In [ ]:
import numpy as np
import pandas as pd

n_steps = 60
n_sims = 100000
dt = 1.0

S0 = tesla_close.iloc[-1]

# matriz de trayectorias: filas = tiempo, columnas = simulaciones
trayectorias = np.zeros((n_steps + 1, n_sims))
trayectorias[0, :] = S0

z = np.random.normal(size=(n_steps, n_sims))

for t in range(1, n_steps + 1):
    trayectorias[t, :] = trayectorias[t-1, :] * np.exp(
        (mu.item() - 0.5 * std**2) * dt + std * np.sqrt(dt) * z[t-1, :]
    )

# pasar a DataFrame con fechas
last_date = tesla_close.index[-1]
future_dates = pd.bdate_range(start=last_date, periods=n_steps+1)

trayectorias_df = pd.DataFrame(trayectorias, index=future_dates)


ValueError: Length of values (100000) does not match length of index (1)

In [ ]:
ejemplo = np.zeros((n_steps + 1,n_sims))
ejemplo[0,:] = s_0
ejemplo

array([[403.98999023, 403.98999023, 403.98999023, ..., 403.98999023,
        403.98999023, 403.98999023],
       [  0.        ,   0.        ,   0.        , ...,   0.        ,
          0.        ,   0.        ],
       [  0.        ,   0.        ,   0.        , ...,   0.        ,
          0.        ,   0.        ],
       ...,
       [  0.        ,   0.        ,   0.        , ...,   0.        ,
          0.        ,   0.        ],
       [  0.        ,   0.        ,   0.        , ...,   0.        ,
          0.        ,   0.        ],
       [  0.        ,   0.        ,   0.        , ...,   0.        ,
          0.        ,   0.        ]])

,log retornos
Date,
2025-01-02,NaN
2025-01-03,0.082156
2025-01-06,0.001486
2025-01-07,-0.040603
2025-01-08,0.001471
...,...
2025-11-11,-0.012600
2025-11-12,-0.020518
2025-11-13,-0.066442
